# Polymarket crypto-market calibration

Scores Polymarket crypto prediction mids against **settled outcomes** (`pm_resolved.yes_outcome` from `outcomePrices`), using the mid ~24h before resolution as the prediction. Backfilled by `backfill_history.py`, so this is populated now — the honest 'is this prediction market well-calibrated' test.

In [ ]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
print('resolved markets:', con.execute('select count(*) from pm_resolved where yes_outcome is not null').fetchone()[0])

## Build the scoring set

Prediction = positive-outcome ('Yes'/'Up') mid ~24h before `end_date`; outcome = settled `yes_outcome` (≥0.5 → 1).

In [ ]:
res = con.execute('''SELECT condition_id, question, yes_outcome, end_date FROM pm_resolved
  WHERE yes_outcome IS NOT NULL AND end_date IS NOT NULL''').df()
rows=[]
for r in res.itertuples():
    h = con.execute('''SELECT t, mid FROM pm_price_hist WHERE condition_id=? AND outcome IN ('Yes','Up')
        ORDER BY t''', [r.condition_id]).df()
    if h.empty: continue
    cutoff = r.end_date - pd.Timedelta(hours=24)
    before = h[h.t <= cutoff]
    pred = float(before.mid.iloc[-1]) if len(before) else float(h.mid.iloc[0])
    rows.append({'question': r.question[:46], 'pred_yes': pred, 'outcome': int(r.yes_outcome>=0.5)})
sf = pd.DataFrame(rows)
print(f'{len(sf)} scored markets | base rate {sf.outcome.mean():.3f}')
sf.head(12)

## Calibration & Brier

Brier = mean((pred−outcome)²). On the diagonal = well-calibrated.

In [ ]:
if len(sf) >= 10:
    brier = float(np.mean((sf.pred_yes - sf.outcome)**2))
    print(f'Brier {brier:.4f} over {len(sf)} markets (a 50/50 coin = 0.25)')
    from sklearn.calibration import calibration_curve
    nb_bins = min(8, max(3, len(sf)//5))
    fp, mp = calibration_curve(sf.outcome, sf.pred_yes, n_bins=nb_bins, strategy='quantile')
    plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'k--',label='perfect')
    plt.plot(mp, fp, 'o-', label='Polymarket mid (24h pre-close)')
    plt.xlabel('predicted P (mid)'); plt.ylabel('realized frequency')
    plt.title(f'Polymarket crypto calibration (n={len(sf)})'); plt.legend(); plt.show()
else:
    print(f'only {len(sf)} scored — need >=10. Re-run backfill_history.py --only polymarket.')

## Example: a resolved market's mid converging to its outcome

In [ ]:
ex = con.execute('''SELECT p.question, p.t, p.mid FROM pm_price_hist p
  JOIN pm_resolved r ON r.condition_id=p.condition_id
  WHERE p.outcome IN ('Yes','Up') AND r.yes_outcome IS NOT NULL
  AND p.condition_id = (SELECT condition_id FROM pm_price_hist WHERE outcome IN ('Yes','Up')
                        GROUP BY condition_id ORDER BY count(*) DESC LIMIT 1)
  ORDER BY p.t''').df()
if len(ex):
    plt.figure(figsize=(11,4)); plt.plot(ex.t, ex.mid, lw=.9)
    plt.ylim(-0.02,1.02); plt.ylabel('mid (implied P)'); plt.title(ex.question.iloc[0][:80]); plt.show()
else: print('no price history yet')